# Chapter 02 — Workflow design and delegation · Practice

**Goal:** redesign Fernhill Outdoor's weekly feedback digest step by step: map it, decide who does each step, place checkpoints, pick the effort per step, write an agent brief, then run the redesigned workflow end to end on a Claude stand-in and an OpenAI stand-in through one small interface, and watch the run log shrink as recurring fixes get promoted into the setup.

- Theory: [README.md](README.md) (read section 2.x before running the matching section here)
- Estimated time: 1.5–2 hours
- Estimated API cost: **$0** by default. With `RUN_LIVE = True` the two live cells send a few short requests to `claude-sonnet-5-5` and `gpt-6-luna` (well under $0.05 in total).
- Prerequisites: the venv from [00-prerequisites](../../00-prerequisites/README.md); `.env` keys only if you go live.

**How it runs offline.** Every model call goes through one tiny interface, `llm.complete(system, prompt, effort)`. `ScriptedLLM` answers from simple rules (with a few realistic mistakes built in), so every cell runs for free and gives the same result each time. `ClaudeLLM` and `OpenAILLM` implement the same interface on the real SDKs and are used only when `RUN_LIVE = True`.

In [ ]:
import json
import os
import random
import re
from collections import Counter
from dataclasses import dataclass, field
from typing import Callable, Protocol

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
CLAUDE_MODEL = os.getenv("CLAUDE_MODEL", "claude-sonnet-5-5")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")
RUN_LIVE = False   # opt-in: True runs the two live cells (needs both keys, costs cents)
print("CLAUDE_MODEL =", CLAUDE_MODEL, "| OPENAI_MODEL =", OPENAI_MODEL, "| RUN_LIVE =", RUN_LIVE)

### Offline toolkit and data (run this once)

The feedback data is synthetic: 16 comments with the theme a careful human would give each one (`label`). Some contain personal data on purpose (emails, phone numbers, order numbers), and two are ambiguous (they touch two themes).

In [ ]:
THEMES = ["delivery", "sizing", "quality", "returns", "app", "price"]
PRODUCTS = ["Trail-X", "Ridge 2"]

BASE = [  # (id, channel, text, label, ambiguous)
    ("c01", "ticket", "Parcel arrived 6 days late, order #A10231. Email me at jo.k@example.com", "delivery", False),
    ("c02", "survey", "The Trail-X jacket runs small, I had to size up twice.", "sizing", False),
    ("c03", "ticket", "Zip broke after two hikes. Call me on +44 7700 900123.", "quality", False),
    ("c04", "survey", "Returns label never came, order #A10377.", "returns", False),
    ("c05", "ticket", "App crashes when I open my order history.", "app", False),
    ("c06", "survey", "Delivery was quick and the box was intact.", "delivery", False),
    ("c07", "ticket", "Ridge 2 boots are a full size bigger than the chart says.", "sizing", False),
    ("c08", "survey", "Price went up 20% since spring, feels steep.", "price", False),
    ("c09", "ticket", "Stitching came loose on the Trail-X after a week.", "quality", False),
    ("c10", "survey", "Courier left it in the rain, the item was soaked and the seams leak.", "delivery", True),
    ("c11", "ticket", "Refund took three weeks, email sam.r@example.com with updates.", "returns", False),
    ("c12", "survey", "Checkout in the app keeps timing out.", "app", False),
    ("c13", "ticket", "Late again, second parcel this month delayed.", "delivery", False),
    ("c14", "survey", "Sizing chart is confusing, too small and too pricey for what it is.", "sizing", True),
    ("c15", "ticket", "Fabric pilled after one wash.", "quality", False),
    ("c16", "survey", "The app login code never arrives.", "app", False),
]


def week_feedback(week: int) -> list[dict]:
    # Week 2 is the odd one out: the survey export failed, so only tickets arrive.
    rows = [dict(id=i, channel=ch, text=t, label=lab, ambiguous=amb) for i, ch, t, lab, amb in BASE]
    return [r for r in rows if not (week == 2 and r["channel"] == "survey")]


class LLM(Protocol):
    name: str
    def complete(self, system: str, prompt: str, effort: str = "medium") -> str: ...


@dataclass
class ScriptedLLM:
    # Offline stand-in: `brain(system, prompt, effort)` returns the reply text. Records every call.
    name: str
    brain: Callable[[str, str, str], str]
    calls: list = field(default_factory=list)

    def complete(self, system: str, prompt: str, effort: str = "medium") -> str:
        self.calls.append({"system": system, "prompt": prompt, "effort": effort})
        return self.brain(system, prompt, effort)


def parse_json(text: str):
    # Pull the first JSON object out of a reply (live models sometimes add prose around it).
    match = re.search(r"\{.*\}", text, re.S)
    if not match:
        raise ValueError(f"no JSON object in reply: {text[:80]!r}")
    return json.loads(match.group(0))


print(len(BASE), "comments |", Counter(r[3] for r in BASE))

In [ ]:
# The scripted "model". Its mistakes are the ones the run log will catch later:
#  - tags a product name as a theme unless the product list is in the context
#  - at effort "low" it gets ambiguous comments wrong (picks the second theme)
#  - paraphrases one quote unless told to copy quotes verbatim
#  - writes counts as prose unless the template asks for a table
KEYWORDS = {"delivery": ["parcel", "late", "delivery", "courier", "delayed"], "sizing": ["size", "sizing"],
            "quality": ["zip", "stitching", "fabric", "broke", "seams"], "returns": ["return", "refund"],
            "app": ["app"], "price": ["price", "pricey"]}
SECOND_THEME = {"c10": "quality", "c14": "price"}


def comment_lines(prompt: str) -> list[tuple[str, str]]:
    return re.findall(r"^(c\d\d): (.*)$", prompt, re.M)


def fake_brain(system: str, prompt: str, effort: str) -> str:
    context = system + prompt
    if "TASK: tag" in system:
        tags = {}
        for cid, text in comment_lines(prompt):
            low = text.lower()
            theme = next((t for t, kws in KEYWORDS.items() if any(k in low for k in kws)), "other")
            if cid in SECOND_THEME and effort == "low":
                theme = SECOND_THEME[cid]
            if "Trail-X" in text and "Products:" not in context:
                theme = "trail-x"          # mistakes a product for a theme
            tags[cid] = theme
        return json.dumps(tags)
    if "TASK: quotes" in system:
        themes = re.search(r"THEMES: (.*)", prompt).group(1).split(", ")
        rows = comment_lines(prompt)
        quotes = {t: [txt for cid, txt in rows if f"[{t}]" in txt][:2] for t in themes}
        quotes = {t: [q.replace(f" [{t}]", "") for q in qs] for t, qs in quotes.items()}
        if "verbatim" not in context and quotes.get(themes[0]):
            q = quotes[themes[0]][0]
            quotes[themes[0]][0] = "Customers say " + q[0].lower() + q[1:]   # a paraphrase
        return json.dumps(quotes)
    if "TASK: draft" in system:
        counts = json.loads(re.search(r"COUNTS: (\{.*\})", prompt).group(1))
        if "| Theme | Count |" in context:
            body = "| Theme | Count |\n|---|---|\n" + "\n".join(f"| {t} | {n} |" for t, n in counts.items())
        else:
            body = ", ".join(f"{t} came up {n} times" for t, n in counts.items()) + "."
        return f"# Weekly feedback digest\n\n{body}\n\nQuotes are attached below."
    if "TASK: plan" in system:
        return json.dumps(PLAN_REPLY)
    return "I am a scripted stand-in and did not understand the task."


fake_claude = ScriptedLLM("claude (offline)", fake_brain)
fake_openai = ScriptedLLM("openai (offline)", fake_brain)
print(fake_claude.complete("TASK: tag", "c02: The Trail-X jacket runs small", "low"))

In [ ]:
# Live adapters: the same interface on the real SDKs. Only constructed when RUN_LIVE is True.
class ClaudeLLM:
    name = "claude (live)"

    def __init__(self, model: str = CLAUDE_MODEL):
        import anthropic
        self.client, self.model = anthropic.Anthropic(), model

    def complete(self, system: str, prompt: str, effort: str = "medium") -> str:
        r = self.client.messages.create(
            model=self.model, max_tokens=4096, system=system,
            output_config={"effort": effort},
            messages=[{"role": "user", "content": prompt}])
        return next((b.text for b in r.content if b.type == "text"), "")


class OpenAILLM:
    name = "openai (live)"

    def __init__(self, model: str = OPENAI_MODEL):
        import openai
        self.client, self.model = openai.OpenAI(), model

    def complete(self, system: str, prompt: str, effort: str = "medium") -> str:
        r = self.client.responses.create(
            model=self.model, instructions=system, input=prompt,
            reasoning={"effort": effort})
        return r.output_text


if RUN_LIVE:
    assert os.getenv("ANTHROPIC_API_KEY") and os.getenv("OPENAI_API_KEY"), "Set both keys in .env"

## 2.1 From prompts to workflows

A good first workflow **recurs**, has a **clear output**, takes **real time**, and has a **quality bar** someone can judge. [Theory](README.md#21-from-prompts-to-workflows)

In [ ]:
CANDIDATES = {
    "weekly feedback digest": dict(recurs=True, clear_output=True, hours_per_month=16, judgeable=True),
    "one-off rebrand strategy": dict(recurs=False, clear_output=True, hours_per_month=40, judgeable=False),
    "monthly invoice exceptions": dict(recurs=True, clear_output=True, hours_per_month=10, judgeable=True),
    "'be more innovative'": dict(recurs=False, clear_output=False, hours_per_month=0, judgeable=False),
}


def is_good_candidate(c: dict, min_hours: int = 4) -> bool:
    return c["recurs"] and c["clear_output"] and c["judgeable"] and c["hours_per_month"] >= min_hours


for name, c in CANDIDATES.items():
    print(f"{'GOOD' if is_good_candidate(c) else 'skip':4}  {name}")

## 2.2 Map the workflow: steps, inputs and outputs

Write the workflow down as it runs today, with an input, output, owner and "done when" per step. [Theory](README.md#22-map-the-workflow-steps-inputs-and-outputs)

In [ ]:
@dataclass
class Step:
    name: str
    input: str
    output: str
    done_when: str
    owner: str = "analyst"


FERNHILL = [
    Step("export", "help desk, survey tool", "feedback rows", "all channels present"),
    Step("redact", "feedback rows", "redacted rows", "no emails, phones or order numbers"),
    Step("tag", "redacted rows", "theme per row", "every row has a theme from the list"),
    Step("count", "theme per row", "trend table", "totals match the row count"),
    Step("quotes", "tagged rows", "2 quotes per top theme", "quotes are verbatim"),
    Step("draft", "trend table, quotes", "draft digest", "follows the template"),
    Step("review", "draft, sources", "approved digest", "reviewer signs off"),
    Step("backlog", "approved digest", "1-3 backlog tickets", "product owner decides", owner="product owner"),
    Step("send", "approved digest", "posted message", "posted after approval only"),
]
for i, s in enumerate(FERNHILL, 1):
    print(f"{i}. {s.name:8} {s.input:24} -> {s.output:24} | done when: {s.done_when}")

## 2.3 Use AI to analyze, plan and design the workflow

Ask for a **structured plan** with assumptions and open questions, then check it yourself. The scripted reply below is what a model might return after interviewing you; the checker finds what you must fix or confirm. [Theory](README.md#23-use-ai-to-analyze-plan-and-design-the-workflow)

In [ ]:
PLAN_REPLY = {
    "steps": [
        {"name": "export", "input": "help desk", "output": "rows", "owner": "analyst", "done_when": "all channels"},
        {"name": "summarize", "input": "rows", "output": "digest", "owner": "AI", "done_when": ""},
        {"name": "send", "input": "digest", "output": "post", "owner": "AI", "done_when": "posted"},
    ],
    "assumptions": ["The digest can be posted without review", "Survey comments need no redaction"],
    "open_questions": ["Who is the audience?", "Which themes does the product team track?"],
}

PLAN_SYSTEM = "TASK: plan. Interview the user, then return JSON: steps, assumptions, open_questions."
plan = parse_json(fake_claude.complete(PLAN_SYSTEM, "Weekly feedback digest, details: ..."))


def plan_problems(plan: dict) -> list[str]:
    issues = [f"step '{s['name']}' has no done-when" for s in plan["steps"] if not s.get("done_when")]
    issues += [f"step '{s['name']}' has no human owner" for s in plan["steps"] if s.get("owner") == "AI"]
    issues += [f"check assumption: {a}" for a in plan["assumptions"]]
    return issues


print("\n".join(plan_problems(plan)))

> **Observe:** the plan reads well, but it lumps tagging, counting and drafting into one "summarize" step, gives two steps no human owner and assumes nothing needs review. The model drafts the design; you own it.

## 2.4 Delegate, share or keep: the delegation worksheet

Score each step, then apply the six rules in order. Delegation pays only when **checking is cheaper than doing**. [Theory](README.md#24-delegate-share-or-keep-the-delegation-worksheet)

In [ ]:
def decide(s: dict) -> str:
    if s["rule_based"]:
        return "code"
    if s["owner_decision"] or s["ai_fit"] == "low":
        return "human"
    if s["check_minutes"] >= s["do_minutes"]:
        return "human"
    if s["impact"] == "high" or not s["reversible"]:
        return "shared"
    return "ai"


W = dict(rule_based=False, owner_decision=False, ai_fit="high", impact="low", reversible=True)
WORKSHEET = {
    "export":  {**W, "rule_based": True, "check_minutes": 1, "do_minutes": 10},
    "redact":  {**W, "rule_based": True, "check_minutes": 2, "do_minutes": 30},
    "tag":     {**W, "check_minutes": 10, "do_minutes": 90},
    "count":   {**W, "rule_based": True, "check_minutes": 1, "do_minutes": 15},
    "quotes":  {**W, "impact": "high", "check_minutes": 5, "do_minutes": 20},
    "draft":   {**W, "impact": "high", "check_minutes": 15, "do_minutes": 60},
    "review":  {**W, "owner_decision": True, "check_minutes": 0, "do_minutes": 15},
    "backlog": {**W, "owner_decision": True, "check_minutes": 0, "do_minutes": 20},
    "send":    {**W, "rule_based": True, "reversible": False, "check_minutes": 1, "do_minutes": 2},
}
DECISIONS = {name: decide(s) for name, s in WORKSHEET.items()}
for name, d in DECISIONS.items():
    print(f"{name:8} -> {d}")

> **Try it:** set `draft`'s `check_minutes` to 60 (the reviewer re-reads every ticket to trust the draft). The decision flips to `human`: as designed, delegating the draft saves nothing. The fix is a design change (draft only from the trend table, with ticket ids to check), not a stronger model.

## 2.5 Review checkpoints

Checkpoints go **before irreversible or external actions**, **after shared steps**, and **after early steps whose errors spread**. AI-only steps get a **spot check on a sample**. [Theory](README.md#25-review-checkpoints)

In [ ]:
EXTERNAL = {"send"}
SPREADS = {"tag"}          # wrong tags silently skew every count after them


def place_checkpoints(decisions: dict) -> dict[str, str]:
    out = {}
    for name, d in decisions.items():
        if name in EXTERNAL or not WORKSHEET[name]["reversible"]:
            out[name] = "before: approval required"
        elif d == "shared":
            out[name] = "after: approve the AI draft"
        elif name in SPREADS:
            out[name] = "after: spot-check a sample"
    return out


def spot_check(rows: list[dict], tags: dict, n: int = 5, seed: int = 0) -> float:
    sample = random.Random(seed).sample(rows, min(n, len(rows)))
    return sum(tags[r["id"]] == r["label"] for r in sample) / len(sample)


for name, why in place_checkpoints(DECISIONS).items():
    print(f"{name:8} {why}")

## 2.6 Model and reasoning effort per step: quality, speed and cost

Measure each effort level on labeled examples, then pick the **cheapest option that meets the quality bar**, counting human minutes. The prices and minutes below are illustrative numbers for this exercise. [Theory](README.md#26-model-and-reasoning-effort-per-step-quality-speed-and-cost)

In [ ]:
TAG_SYSTEM = "TASK: tag. Give each comment exactly one theme from: " + ", ".join(THEMES) + \
             ". Products: " + ", ".join(PRODUCTS) + " (products are never themes). Return JSON {id: theme}."
rows = week_feedback(1)
prompt = "\n".join(f"{r['id']}: {r['text']}" for r in rows)

OPTIONS = {"low": (0.02, 1), "medium": (0.05, 3), "high": (0.15, 8)}   # ($ per run, minutes elapsed)
HOURLY = 60          # $ per hour of reviewer time
results = {}
for effort, (usd, minutes) in OPTIONS.items():
    tags = parse_json(fake_claude.complete(TAG_SYSTEM, prompt, effort))
    errors = sum(tags[r["id"]] != r["label"] for r in rows)
    human_min = 10 + 2 * errors                     # spot check + fixing each wrong tag
    results[effort] = dict(quality=1 - errors / len(rows), total=usd + human_min * HOURLY / 60, minutes=minutes)
    print(f"{effort:6} accuracy {results[effort]['quality']:.0%}  model ${usd:.2f}  "
          f"human {human_min} min  total ${results[effort]['total']:.2f}  elapsed {minutes} min")


def cheapest_meeting_bar(results: dict, bar: float) -> str | None:
    ok = [(r["total"], r["minutes"], e) for e, r in results.items() if r["quality"] >= bar]
    return min(ok)[2] if ok else None


for bar in (0.85, 0.95):
    print(f"bar {bar:.0%}: choose {cheapest_meeting_bar(results, bar)}")

> **Observe:** `low` passes an 85% bar and is the cheapest *model* setting, but its two wrong tags cost more reviewer time than it saves, so `medium` is the cheapest *workflow*. `high` adds cost and waiting time for no quality gain. Human minutes dominate the total, which is why you measure on real examples instead of picking the cheapest or the strongest setting by habit. The mini-project below tags at `medium`.

In [ ]:
# live, costs money: one short request per vendor at two effort levels (RUN_LIVE must be True)
if RUN_LIVE:
    question = "In one sentence: is 'the zip broke after two hikes' about delivery, sizing or quality?"
    for llm in (ClaudeLLM(), OpenAILLM()):
        for effort in ("low", "high"):
            print(f"{llm.name:14} {effort:5}", llm.complete("Answer briefly.", question, effort)[:150])
else:
    print("Skipped (RUN_LIVE = False).")

## 2.7 From asking to assigning: the agent brief

A brief carries what you would have said along the way: goal, context, deliverable, done-when, boundaries and stop rules. [Theory](README.md#27-from-asking-to-assigning-the-agent-brief)

In [ ]:
REQUIRED = ["goal", "context", "deliverable", "done_when", "allowed", "needs_approval", "stop_and_ask"]
VAGUE = ("look into", "improve", "handle", "deal with")


def brief_gaps(brief: dict) -> list[str]:
    gaps = [f"missing: {f}" for f in REQUIRED if not brief.get(f)]
    if any(v in brief.get("goal", "").lower() for v in VAGUE):
        gaps.append("goal is an activity, not an outcome")
    return gaps


ASKED = {"goal": "Look into last week's feedback and improve the digest"}
BRIEF = {
    "goal": "A draft digest the product team can act on in 5 minutes",
    "context": ["redacted export", "last week's digest", "theme list", "product list"],
    "deliverable": "Markdown following the template; counts in a table",
    "done_when": "every number traces to the data; 2 verbatim quotes per top theme",
    "allowed": ["read /shared/feedback", "write /drafts"],
    "needs_approval": ["post to any channel", "share outside /drafts"],
    "never": ["contact customers", "open unredacted data"],
    "stop_and_ask": ["a theme is not on the list", "a channel is missing from the export"],
    "budget": "15 minutes or 30 steps, then report progress",
}
print("asked :", brief_gaps(ASKED))
print("brief :", brief_gaps(BRIEF) or "no gaps")

## 2.8 Context and the first draft

Standing context lives in the **setup** (a project's instructions and files), not in each prompt. Then **draft small**: one week first, review against the brief, fix the setup, try other weeks. [Theory](README.md#28-context-and-the-first-draft)

In [ ]:
@dataclass
class Setup:
    # What a Claude Project or ChatGPT Project would hold: standing instructions and knowledge.
    instructions: list[str] = field(default_factory=lambda: [f"Themes: {', '.join(THEMES)}."])
    template: str = "# Weekly feedback digest\n<counts>\n<quotes>"

    def system(self, task: str) -> str:
        return f"TASK: {task}.\n" + "\n".join(self.instructions) + f"\nTemplate:\n{self.template}"


first = Setup()
print(first.system("draft"))

## 2.10 Responsible use inside the workflow

(Before the mini-project, because the pipeline needs it.) Personal data is removed **in code, before** any AI step, and the check is code too. [Theory](README.md#210-responsible-use-inside-the-workflow)

In [ ]:
PII = [(re.compile(r"[\w.]+@[\w.]+\.\w+"), "[email]"),
       (re.compile(r"\+?\d[\d ]{8,}\d"), "[phone]"),
       (re.compile(r"#A\d{5}"), "[order]")]


def redact(text: str) -> str:
    for pattern, token in PII:
        text = pattern.sub(token, text)
    return text


def assert_no_pii(texts: list[str]) -> None:
    leaks = [t for t in texts if any(p.search(t) for p, _ in PII)]
    assert not leaks, f"personal data would reach the AI step: {leaks[:2]}"


redacted = [redact(r["text"]) for r in week_feedback(1)]
assert_no_pii(redacted)
print(redacted[0], "|", redacted[2])

## 2.9 + 2.11 Mini-project: run, review, improve, on both vendors

The redesigned workflow, end to end: **code** exports, redacts, counts and sends; **AI** tags (at the effort chosen in 2.6, spot-checked); **shared** steps (quotes, draft) are reviewed against a checklist; the **human** product owner picks backlog items. Every model call goes through `llm.complete`, so the same function runs on the Claude and OpenAI stand-ins (or the live adapters). [Theory](README.md#29-improve-reuse-and-share-the-workflow), [side by side](README.md#211-claude-and-openai-side-by-side)

`review()` simulates the human reviewer's checklist. Each correction is logged with its kind: `context`, `format`, `verification` or `one_off`.

In [ ]:
def review(rows, tags, quotes, draft, setup) -> list[tuple[str, str]]:
    # Simulated reviewer working from the done-when checklist. Returns (kind, note) corrections.
    fixes = []
    if {r["channel"] for r in rows} != {"ticket", "survey"}:
        fixes.append(("one_off", "survey export missing this week"))
    if any(t not in THEMES for t in tags.values()):
        fixes.append(("context", "a product name was tagged as a theme"))
    sources = " ".join(redact(r["text"]) for r in rows)
    if any(q not in sources for qs in quotes.values() for q in qs):
        fixes.append(("verification", "a quote is not verbatim"))
    if "| Theme | Count |" not in draft:
        fixes.append(("format", "counts written as prose, not a table"))
    return fixes


def run_workflow(llm: LLM, week: int, setup: Setup, channel: list) -> dict:
    rows = week_feedback(week)                                              # 1 export (code)
    clean = {r["id"]: redact(r["text"]) for r in rows}                      # 2 redact (code)
    assert_no_pii(list(clean.values()))
    listing = "\n".join(f"{cid}: {t}" for cid, t in clean.items())
    tags = parse_json(llm.complete(setup.system("tag"), listing, "medium")) # 3 tag (AI)
    counts = dict(Counter(tags.values()).most_common())                     # 4 count (code)
    top = [t for t in counts if t in THEMES][:2]
    tagged = "\n".join(f"{cid}: {clean[cid]} [{tags[cid]}]" for cid in clean)
    quotes = parse_json(llm.complete(setup.system("quotes"),                # 5 quotes (shared)
                                     f"THEMES: {', '.join(top)}\n{tagged}", "medium"))
    draft = llm.complete(setup.system("draft"), f"COUNTS: {json.dumps(counts)}", "medium")  # 6 draft
    fixes = review(rows, tags, quotes, draft, setup)                       # 7 review (human)
    backlog = top[:1]                                                       # 8 product owner decides
    channel.append(f"week {week}: digest posted")                           # 9 send (code), after review
    return {"week": week, "vendor": llm.name, "kinds": [k for k, _ in fixes], "fixes": fixes,
            "backlog": backlog, "spot_check": spot_check(rows, tags)}


channel: list[str] = []
for llm in (fake_claude, fake_openai):
    out = run_workflow(llm, 1, Setup(), channel)
    print(f"{out['vendor']:17} corrections: {[n for _, n in out['fixes']]}")

> **Observe:** the same workflow code runs on both stand-ins; only the adapter changes. The first runs need three corrections: missing context (product names), a verification miss (paraphrased quote) and a format miss.

In [ ]:
TARGETS = {"context": "project instructions/knowledge", "format": "template + example",
           "verification": "checklist item or code check"}


def promotions(log: list[dict], threshold: int = 3, window: int = 5) -> dict[str, str]:
    seen = Counter(k for run in log[-window:] for k in set(run["kinds"]))
    return {k: TARGETS[k] for k, n in seen.items() if k in TARGETS and n >= threshold}


def apply(setup: Setup, promoted: dict) -> Setup:
    s = Setup(list(setup.instructions), setup.template)
    if "context" in promoted:
        s.instructions.append(f"Products: {', '.join(PRODUCTS)} (products are never themes).")
    if "verification" in promoted:
        s.instructions.append("Copy quotes verbatim from the comments; never paraphrase.")
    if "format" in promoted:
        s.template = "# Weekly feedback digest\n| Theme | Count |\n|---|---|\n<quotes>"
    return s


setup, log, channel = Setup(), [], []
for week in range(1, 7):
    run = run_workflow(fake_claude, week, setup, channel)
    log.append(run)
    promoted = promotions(log)
    if promoted and setup.template == Setup().template:   # promote once, then keep running
        setup = apply(setup, promoted)
        print(f"after week {week}: promoted -> {promoted}")
    print(f"week {week}: corrections {run['kinds'] or 'none'}")

> **Observe:** weeks 1–3 need the same three fixes; after week 3 they are promoted into the setup and weeks 4–6 run clean. The week-2 survey outage is logged as a one-off and promotes nothing.

Now share it: a **workflow card** a teammate could run from.

In [ ]:
def workflow_card(steps, decisions, checkpoints, setup, log) -> str:
    lines = ["WORKFLOW CARD: weekly feedback digest", "Trigger: Monday 09:00 | Owner: analyst", ""]
    for s in steps:
        cp = checkpoints.get(s.name, "")
        lines.append(f"- {s.name:8} {decisions[s.name]:7} done when: {s.done_when}" + (f" | {cp}" if cp else ""))
    recent = log[-3:]
    lines += ["", "Standing context:", *[f"  {i}" for i in setup.instructions],
              f"Last 3 runs: {sum(len(r['kinds']) for r in recent)} corrections",
              "Known limits: tags spot-checked on 5 rows a week; rare themes can be missed."]
    return "\n".join(lines)


print(workflow_card(FERNHILL, DECISIONS, place_checkpoints(DECISIONS), setup, log))

In [ ]:
# live, costs money: run the tagging step of the improved setup on both real vendors (RUN_LIVE must be True)
if RUN_LIVE:
    sample = {r["id"]: redact(r["text"]) for r in week_feedback(1)[:5]}
    listing = "\n".join(f"{cid}: {t}" for cid, t in sample.items())
    system = setup.system("tag") + "\nReturn only a JSON object mapping comment id to theme."
    for llm in (ClaudeLLM(), OpenAILLM()):
        print(llm.name, parse_json(llm.complete(system, listing, "medium")))
else:
    print("Skipped (RUN_LIVE = False).")

**Try it:** design your own recurring workflow: fill in `FERNHILL`-style steps and a `WORKSHEET`, run `decide` and `place_checkpoints`, and write a `BRIEF` that passes `brief_gaps`. Then test yourself in [02_homework.ipynb](02_homework.ipynb).

[← Chapter README](README.md)